In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import CRS, Geod, Transformer
from shapely.geometry import Point
from shapely.ops import nearest_points

warnings.filterwarnings("ignore", category=UserWarning)

POINTS_PANEL = Path("../data/study_grid_cells.csv")
COASTLINE_SHP = Path("../data/raw/coastlines/lines.shp")
OUT_BASE = Path("../outputs/coast_distance")

EXPECTED_POINTS = 652
START_RADIUS_DEG = 2.0
MAX_RADIUS_DEG = 32.0
SUSPECT_RING_KM = 5.0

EXCLUDE_FEATURES = [
    dict(name="Kinshasa", bbox=(15.30, -4.38, 15.33, -4.36), max_len_km=0.2, affects="Kinshasa_DRC"),
    dict(name="Bihar inland ponds", bbox=(85.5, 26.5, 86.2, 27.0), max_len_km=5.0, affects="Delhi_India"),
    dict(name="Lake Victoria", bbox=(31.5, -3.5, 35.0, 1.0), max_len_km=5.0, affects="Nairobi_Kenya"),
]

GEOD = Geod(ellps="WGS84")
KM_PER_DEG = 111.32


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def load_points():
    p = pd.read_csv(POINTS_PANEL, usecols=["city", "cell_id", "lat", "lon"]).drop_duplicates()
    p = p.sort_values(["city", "cell_id"]).reset_index(drop=True)
    require(len(p) == EXPECTED_POINTS and not p.duplicated(["city", "cell_id"]).any(), "Grid-cell count or keys abnormal")
    return p


def read_lines(lon0, lat0, r):
    bbox = (lon0 - r, max(lat0 - r, -90), lon0 + r, min(lat0 + r, 90))
    g = gpd.read_file(COASTLINE_SHP, bbox=bbox)
    require(g.crs is not None and g.crs.to_epsg() == 4326, "Coastline CRS is not EPSG:4326")
    return g, bbox


def edge_distance_km(lon, lat, bbox):
    x0, y0, x1, y1 = bbox
    cos_lat = np.cos(np.radians(max(abs(y0), abs(y1))))
    dx = min(lon - x0, x1 - lon) * KM_PER_DEG * cos_lat
    dy = min(lat - y0, y1 - lat) * KM_PER_DEG
    return min(dx, dy)


def drop_excluded(city, lines, aeqd):
    if not len(lines):
        return lines, []
    L = lines.to_crs(aeqd)
    b = lines.geometry.bounds
    ring = lines.geometry.is_ring.to_numpy()
    len_km = L.length.to_numpy() / 1000
    keep = np.ones(len(lines), bool)
    dropped = []
    for e in EXCLUDE_FEATURES:
        x0, y0, x1, y1 = e["bbox"]
        hit = (ring & (len_km < e["max_len_km"])
               & (b.minx >= x0).to_numpy() & (b.maxx <= x1).to_numpy()
               & (b.miny >= y0).to_numpy() & (b.maxy <= y1).to_numpy())
        for i in np.flatnonzero(hit & keep):
            dropped.append(dict(rule=e["name"], read_for=city, length_m=float(len_km[i] * 1000),
                                bounds=[round(float(v), 5) for v in lines.geometry.iloc[i].bounds]))
        keep &= ~hit
    return lines.loc[keep].reset_index(drop=True), dropped


def city_distances(city, p):
    lon0, lat0 = float(p.lon.mean()), float(p.lat.mean())
    aeqd = CRS.from_proj4(f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m")
    to_local = Transformer.from_crs(4326, aeqd, always_xy=True)
    to_ll = Transformer.from_crs(aeqd, 4326, always_xy=True)
    r = START_RADIUS_DEG
    while True:
        lines, bbox = read_lines(lon0, lat0, r)
        lines, dropped = drop_excluded(city, lines, aeqd)
        if len(lines):
            L = lines.to_crs(aeqd)
            union = L.geometry.union_all()
            rows = []
            for q in p.itertuples():
                pt = Point(*to_local.transform(q.lon, q.lat))
                near = nearest_points(pt, union)[1]
                nlon, nlat = to_ll.transform(near.x, near.y)
                _, _, dist_m = GEOD.inv(q.lon, q.lat, nlon, nlat)
                j = int(np.argmin(L.distance(near).to_numpy()))
                rows.append(dict(city=city, cell_id=int(q.cell_id), lat=q.lat, lon=q.lon,
                                 coast_dist_km=dist_m / 1000, nearest_coast_lon=nlon, nearest_coast_lat=nlat,
                                 nearest_feature_len_km=float(L.length.iloc[j]) / 1000,
                                 nearest_feature_closed=bool(L.geometry.iloc[j].is_ring),
                                 search_radius_deg=r))
            out = pd.DataFrame(rows)
            margin = min(edge_distance_km(q.lon, q.lat, bbox) for q in p.itertuples())
            if out.coast_dist_km.max() < margin:
                return out, dropped
        require(r < MAX_RADIUS_DEG, f"{city}: no reliable nearest coast found even with a search radius up to {MAX_RADIUS_DEG}°")
        r *= 2


def run_coast():
    pts = load_points()
    cities = sorted(pts.city.unique())
    out_dir = OUT_BASE
    out_dir.mkdir(parents=True, exist_ok=True)

    parts, dropped_all = [], []
    for city in cities:
        df, dropped = city_distances(city, pts[pts.city == city])
        parts.append(df)
        dropped_all += dropped
        print(f"{city:32s} {df.coast_dist_km.min():8.2f} – {df.coast_dist_km.max():8.2f} km"
              f"  (search radius {df.search_radius_deg.iloc[0]:.0f}°)", flush=True)
    res = pd.concat(parts, ignore_index=True)
    res["needs_review"] = res.nearest_feature_closed & (res.nearest_feature_len_km < SUSPECT_RING_KM)
    require(res.coast_dist_km.notna().all() and (res.coast_dist_km >= 0).all(), "Distances missing or negative")
    require(len(res) == EXPECTED_POINTS, "Wrong number of output grid cells")
    for e in EXCLUDE_FEATURES:
        if e["affects"] in cities and not any(d["rule"] == e["name"] and d["read_for"] == e["affects"] for d in dropped_all):
            print(f"[note] {e['name']}: no feature was removed when reading {e['affects']}; "
                  "the coastline data may differ from the version used in the paper", flush=True)
    dropped_all = list({(d["rule"], tuple(d["bounds"])): d for d in dropped_all}.values())

    res.to_csv(out_dir / "coast_distance_652.csv", index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(out_dir / "coast_distance_652.csv", float_precision="round_trip")
    pd.testing.assert_frame_equal(res, back, check_dtype=False, check_exact=True)
    summary = dict(
        status="calculation_complete", points=int(len(res)), cities=int(res.city.nunique()),
        excluded_features=dropped_all, needs_review_points=int(res.needs_review.sum()),
        source=str(COASTLINE_SHP))
    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output folder:", out_dir)
    return res


COAST = run_coast()

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Geod

warnings.filterwarnings("ignore", category=UserWarning)

CBD_SHP = Path("../data/cbd/cbd.shp")
POINTS_PANEL = Path("../data/study_grid_cells.csv")
OUT_BASE = Path("../outputs/cbd_distance")

EXPECTED_POINTS = 652
CBD_TYPES = ["modern CBD", "historic core"]
ALL_TYPES = CBD_TYPES + ["secondary centre"]
GEOD = Geod(ellps="WGS84")


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z]", "", s.lower())


def geodesic_km(lat, lon, plat, plon):
    n = len(plat)
    _, _, d = GEOD.inv(np.full(n, lon), np.full(n, lat), plon, plat)
    return d / 1000


def nearest(lat, lon, pts):
    d = geodesic_km(lat, lon, pts.y.to_numpy(), pts.x.to_numpy())
    i = int(np.argmin(d))
    return float(d[i]), pts.name1.iloc[i]


def load():
    cbd = gpd.read_file(CBD_SHP)
    require(cbd.crs is not None and cbd.crs.to_epsg() == 4326, "CBD point CRS is not EPSG:4326")
    require(set(cbd.type1) <= set(ALL_TYPES), f"Unknown type1: {set(cbd.type1) - set(ALL_TYPES)}")
    require(np.allclose(cbd.geometry.x, cbd.x) and np.allclose(cbd.geometry.y, cbd.y), "x/y fields do not match the geometry")
    pts = (pd.read_csv(POINTS_PANEL, usecols=["city", "cell_id", "lat", "lon"]).drop_duplicates()
           .sort_values(["city", "cell_id"]).reset_index(drop=True))
    require(len(pts) == EXPECTED_POINTS and not pts.duplicated(["city", "cell_id"]).any(), "Study grid cells abnormal")
    key = {c: norm(re.sub(r"_[A-Za-z]+$", "", c)) for c in pts.city.unique()}
    alias = {"hochiminh": "hochiminhcity"}
    key = {c: alias.get(k, k) for c, k in key.items()}
    cbd["key"] = cbd.city1.map(norm)
    missing = [c for c, k in key.items() if not cbd.key.eq(k).any()]
    require(not missing, f"These cities have no CBD point: {missing}")
    return cbd, pts, key


def run_cbd():
    cbd, pts, key = load()
    out_dir = OUT_BASE
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for city, p in pts.groupby("city"):
        c_cbd = cbd[cbd.key.eq(key[city]) & cbd.type1.isin(CBD_TYPES)]
        require(len(c_cbd) > 0, f"{city}: no modern CBD / historic core point")
        for r in p.itertuples():
            d, name = nearest(r.lat, r.lon, c_cbd)
            rows.append(dict(city=city, cell_id=int(r.cell_id), lat=r.lat, lon=r.lon,
                             dist_to_cbd_km=d, nearest_cbd=name, n_cbd_points=len(c_cbd)))
        print(f"{city:32s} {len(c_cbd)} CBD points", flush=True)
    res = pd.DataFrame(rows).sort_values(["city", "cell_id"]).reset_index(drop=True)
    require(len(res) == EXPECTED_POINTS and res.dist_to_cbd_km.notna().all(), "Output grid-cell count or missing values abnormal")

    res.to_csv(out_dir / "cbd_distance_652.csv", index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(out_dir / "cbd_distance_652.csv", float_precision="round_trip")
    pd.testing.assert_frame_equal(res, back, check_dtype=False, check_exact=True)
    used = cbd[cbd.type1.isin(CBD_TYPES)]
    used.drop(columns="geometry").to_csv(out_dir / "cbd_points_used.csv", index=False, encoding="utf-8-sig")

    summary = dict(
        status="calculation_complete", points=int(len(res)), cities=int(res.city.nunique()),
        cbd_source=str(CBD_SHP), cbd_points=int(len(used)), cbd_types=CBD_TYPES)
    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output folder:", out_dir)
    return res


CBD_DIST = run_cbd()

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import shapely
from rasterio.features import geometry_mask, geometry_window
from shapely.geometry import mapping

warnings.filterwarnings("ignore", category=UserWarning)

LCZ_TIF = Path("../data/raw/lcz/lcz_v3.tif")
POINTS_PANEL = Path("../data/study_grid_cells.csv")
GRID_DIR = Path("../data/grids")
BOUNDARY_SHP = Path("../data/smod_0514.shp")
OUT_BASE = Path("../outputs/lcz")

COVERAGE_MIN = 0.90
EXPECTED_POINTS = 652
PAPER_YEARS = list(range(2016, 2024))
CLASSES = list(range(1, 18))
URBAN, NATURAL = list(range(1, 11)), list(range(11, 18))
METRICS = [f"lcz_{i}" for i in CLASSES] + ["lcz_urban_frac", "lcz_natural_frac", "lcz_dominant", "lcz_n_classes"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z]", "", s.lower())


def city_key(city):
    k = norm(re.sub(r"_[A-Za-z]+$", "", city))
    return {"hochiminh": "hochiminhcity"}.get(k, k)


def load_boundaries():
    b = gpd.read_file(BOUNDARY_SHP)
    require(b.crs is not None and b.crs.to_epsg() == 4326, "Boundary CRS is not EPSG:4326")
    b["key"] = b.name.map(norm)
    require(b.key.is_unique, "Duplicate boundary names after normalisation")
    out, fixed = {}, []
    for r in b.itertuples():
        g = r.geometry
        if not g.is_valid:
            g2 = shapely.make_valid(g)
            g2 = shapely.unary_union([p for p in getattr(g2, "geoms", [g2]) if p.geom_type in ("Polygon", "MultiPolygon")])
            require(abs(g2.area - g.buffer(0).area) <= 1e-6 * g2.area, f"{r.name}: area change after boundary repair is abnormal")
            fixed.append(r.name)
            g = g2
        out[r.key] = g
    return out, fixed


def lcz_stats(src, geom):
    win = geometry_window(src, [mapping(geom)], boundless=True)
    arr = src.read(1, window=win, boundless=True, masked=True)
    inside = geometry_mask([mapping(geom)], out_shape=arr.shape, transform=src.window_transform(win),
                           invert=True, all_touched=False)
    v = np.asarray(arr.data)[inside & ~np.ma.getmaskarray(arr)].astype(int)
    require(np.isin(v, [0] + CLASSES).all(), f"Unknown LCZ code found: {sorted(set(v) - set([0] + CLASSES))}")
    n_study, valid = int(inside.sum()), v[v != 0]
    d = {"lcz_study_pixels": n_study, "lcz_valid_pixels": valid.size,
         "lcz_coverage_fraction": valid.size / n_study if n_study else np.nan}
    if valid.size == 0:
        d.update({f"{m}_observed": np.nan for m in METRICS})
        d["lcz_n_classes_observed"] = 0
        return d
    cnt = np.bincount(valid, minlength=18)
    for i in CLASSES:
        d[f"lcz_{i}_observed"] = cnt[i] / valid.size
    d["lcz_urban_frac_observed"] = cnt[URBAN].sum() / valid.size
    d["lcz_natural_frac_observed"] = cnt[NATURAL].sum() / valid.size
    d["lcz_dominant_observed"] = int(np.argmax(cnt[1:]) + 1)
    d["lcz_n_classes_observed"] = int((cnt[1:] > 0).sum())
    return d


def run_lcz():
    pts = (pd.read_csv(POINTS_PANEL, usecols=["city", "cell_id", "lat", "lon"]).drop_duplicates()
           .sort_values(["city", "cell_id"]).reset_index(drop=True))
    require(len(pts) == EXPECTED_POINTS and not pts.duplicated(["city", "cell_id"]).any(), "Study grid cells abnormal")
    bnd, fixed = load_boundaries()
    out_dir = OUT_BASE
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    with rasterio.open(LCZ_TIF) as src:
        require(src.crs.to_epsg() == 4326 and src.count == 1 and src.nodata == 0, "LCZ raster CRS / band / nodata abnormal")
        for city, p in pts.groupby("city"):
            k = city_key(city)
            require(k in bnd, f"{city}: {k} not found in the SMOD boundaries")
            g = gpd.read_file(GRID_DIR / f"{city}_era5_grid.gpkg")
            require(g.crs.to_epsg() == 4326 and g.cell_id.is_unique, f"{city}: grid abnormal")
            g = g.set_index("cell_id").loc[p.cell_id]
            require(np.allclose(g.lat_center, p.lat, atol=1e-6) and np.allclose(g.lon_center, p.lon, atol=1e-6),
                    f"{city}: grid does not match the study grid-cell coordinates")
            for cid, cell in zip(p.cell_id, g.geometry):
                study = cell.intersection(bnd[k])
                d = dict(city=city, cell_id=int(cid), study_area_frac_deg=study.area / cell.area)
                d.update(lcz_stats(src, study) if not study.is_empty else
                         {"lcz_study_pixels": 0, "lcz_valid_pixels": 0, "lcz_coverage_fraction": np.nan})
                rows.append(d)
            print(f"{city:32s} {len(p):3d} grid cells", flush=True)

    res = pd.DataFrame(rows).sort_values(["city", "cell_id"]).reset_index(drop=True)
    require(len(res) == EXPECTED_POINTS, "Wrong number of output grid cells")
    res["lcz_coverage_pass"] = res.lcz_coverage_fraction.fillna(0) >= COVERAGE_MIN
    for m in METRICS:
        res[m] = res[f"{m}_observed"].where(res.lcz_coverage_pass)
    ok = res.lcz_coverage_pass
    require(np.allclose(res.loc[ok, [f"lcz_{i}" for i in CLASSES]].sum(axis=1), 1.0), "Class fractions do not sum to 1")
    require(np.allclose(res.loc[ok, "lcz_urban_frac"] + res.loc[ok, "lcz_natural_frac"], 1.0), "Built types + natural types ≠ 1")

    panel = pd.concat([res.assign(year=y) for y in PAPER_YEARS], ignore_index=True)
    panel = panel[["city", "cell_id", "year"] + [c for c in res.columns if c not in ("city", "cell_id")]]
    panel = panel.sort_values(["city", "cell_id", "year"]).reset_index(drop=True)

    res.to_csv(out_dir / "lcz_smod_static.csv", index=False, encoding="utf-8-sig", float_format="%.17g")
    panel.to_csv(out_dir / "lcz_smod_2016_2023.csv", index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(out_dir / "lcz_smod_2016_2023.csv", float_precision="round_trip")
    pd.testing.assert_frame_equal(panel, back, check_dtype=False, check_exact=True)

    summary = dict(
        status="calculation_complete", source=str(LCZ_TIF),
        coverage_min=COVERAGE_MIN, points=int(len(res)), pass_points=int(ok.sum()),
        below_threshold=res.loc[~ok, ["city", "cell_id", "lcz_coverage_fraction"]].round(4).values.tolist(),
        smod_geometry_repairs=fixed)
    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output folder:", out_dir)
    return panel


LCZ_PANEL = run_lcz()